# Final Evaluation
This Notebook contains the code for the final test evaluation and error analysis. Test metrics, confusion matrices, inspect missed failures and false alarms

### Imports

In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_fscore_support

from src.data import CLASSES, MODES, PRIORITY, RAW_FEATURES, load_raw, make_preprocessor
from src.splits import load_split
from src.models_mlp import make_mlp
from src.models_rf import make_rf, make_class_weights, resolve_params, load_selected_features
from src.evaluate import (evaluate, evaluate_multi, metrics_at_threshold, metrics_at_cut_multi,
                          summarize, summarize_multi, _fold_metrics, _fold_metrics_multi)

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

ROOT = pathlib.Path.cwd().parent
RESULTS, FIG = ROOT / "results", ROOT / "results" / "figures"

## Fetching Final Hyperparameters

In [ ]:
with open(RESULTS / "best_params.json") as f:
    best = json.load(f)
SELECTED = load_selected_features()   # the 9 features chosen in notebook 02 (RF only)

# decision = threshold (binary) or failure cut-off (multi), chosen on validation
FINAL = {
    ("MLP", "binary"): {**best["mlp_bin"],   "features": None},
    ("RF",  "binary"): {**best["rf_bin"],    "features": SELECTED},
    ("MLP", "multi"):  {**best["mlp_multi"], "features": None},
    ("RF",  "multi"):  {**best["rf_multi"],  "features": SELECTED},
}
pd.DataFrame({k: {"decision": v["decision"], **v["params"]} for k, v in FINAL.items()}).T

> These are the four configurations frozen at the model-selection meeting (T12), loaded from `results/best_params.json` (hyperparameters and decision value, which are rounded just for the display, of each model). \
> Nothing is chosen here, the decision rule of each model was fixed on validation: a probability threshold for binary, and a failure cut-off for the two-stage multiclass rule. \
> The RF uses the 9 selected features (`Type` one-hot encoded, no scaling) and the MLP uses all 11 inputs (its pipeline standardises the numeric features and one-hot encodes `Type`), as in tuning.

## Helper Functions

In [ ]:
encode = make_preprocessor(scale=False).set_output(transform="pandas")   # one-hot Type, no scaling

def inputs(model, X, cfg):
    """MLP: the raw columns (its pipeline scales and encodes them). RF: Type one-hot encoded, then the selected features."""
    return X if model == "MLP" else encode.fit_transform(X)[cfg["features"]]

def build(model, task, y_train):
    p = FINAL[(model, task)]["params"]
    return make_mlp(**p) if model == "MLP" else make_rf(**resolve_params(p, make_class_weights(y_train)))

def predict(fitted, X, task, decision):
    """Same decision rules as in validation: threshold on P(failure), or two-stage cut-off for multi."""
    proba = fitted.predict_proba(X)
    if task == "binary":
        return (proba[:, 1] >= decision).astype(int), proba[:, 1]
    p_fail = 1 - proba[:, 0]
    return np.where(p_fail >= decision, 1 + proba[:, 1:].argmax(axis=1), 0), p_fail

## Scoring on Test Set

In [ ]:
splits = {t: load_split(t) for t in ["binary", "multi"]}
rows, preds, fitted = [], {}, {}

for (model, task), cfg in FINAL.items():
    X_tr, X_te, y_tr, y_te = splits[task]
    X_tr, X_te, d = inputs(model, X_tr, cfg), inputs(model, X_te, cfg), cfg["decision"]

    if task == "binary":
        _, probas = evaluate(build(model, task, y_tr), X_tr, y_tr)
        val = summarize(metrics_at_threshold(probas, d))
    else:
        _, probas = evaluate_multi(build(model, task, y_tr), X_tr, y_tr)
        val = summarize_multi(metrics_at_cut_multi(probas, d))

    m = build(model, task, y_tr).fit(X_tr, y_tr)
    pred, score = predict(m, X_te, task, d)
    fitted[(model, task)], preds[(model, task)] = m, (pred, score)

    if task == "binary":
        te = _fold_metrics(y_te, pred)
        keys = {"accuracy": "accuracy", "precision": "precision", "recall": "recall", "f1": "f1"}
    else:
        te = _fold_metrics_multi(y_te, pred)
        keys = {"accuracy": "accuracy", "precision": "detection_precision", "recall": "detection_recall",
                "f1": "detection_f1", "f1_macro": "f1_macro"}
    for metric, k in keys.items():
        rows.append({"model": model, "task": task, "decision": d, "metric": metric,
                     "val_mean": val[f"{k}_mean"], "val_std": val[f"{k}_std"], "test": te[k]})

final = pd.DataFrame(rows)
final.to_csv(RESULTS / "final_metrics.csv", index=False)
final.round(3)

For each model, the validation metrics come from the same 5×3 repeated stratified CV on the training set used in tuning, reported as mean +/- std. The model is then trained once on the full training set and evaluated once on the hold-out test set.

The RF is the better model in both tasks, on validation (where the choice was made) and on test (F1 of class 1, validation mean +/- std → test):
- **Random Forest**: 0.893 +/- 0.026 → 0.881 in binary, and 0.871 +/- 0.025 → 0.871 in multiclass (detection F1)
- **MLP**: 0.736 +/- 0.047 → 0.729 in binary, and 0.799 +/- 0.039 → 0.768 in multiclass (detection F1)

For the multiclass task, "class 1" means *any failure* (detection: the four failure types merged), and the macro F1 over the five classes is reported as well. The test value is a single run, so it has no std.

Three of the four models land within about one std of their validation score.\
**The exception:** MLP in binary, where recall drops from 0.761 to 0.600, about 2.4 std below, while precision rises from 0.720 to 0.927. Its F1 barely changes (0.736 vs 0.729): on test, the MLP is more conservative than on validation, with fewer false alarms and more missed failures.\
A likely reason is that the MLP's recall varies a lot between folds, and the test set has only 85 failures, so each missed failure costs 1.2 points of recall.

## Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, ((model, task), (pred, _)) in zip(axes.ravel(), preds.items()):
    labels = ["No failure", "Failure"] if task == "binary" else CLASSES
    ConfusionMatrixDisplay.from_predictions(splits[task][3], pred, labels=range(len(labels)),
                                            display_labels=labels, ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{model} – {task} (test)"); ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.savefig(FIG / "test_confusion_matrices.png", dpi=150); plt.show()

**Binary:** 
- The RF misses 15 of 85 failures with only 4 false positives. 
- The MLP misses 34 and raises 4 false alarms. 

**Multi-class:** 
- Both models identify HDF, PWF and OSF well (the RF gets almost all of them)
- TWF is almost never detected: 
  - 10 of 11 missed by the MLP
  - 9 missed plus 1 confused with OSF by the RF. 

Confusions between failure types are rare. Almost all errors are between "No failure" and some failure.

## Per-class Test Metrics (Multiclass)

In [ ]:
rows = []
for model in ["MLP", "RF"]:
    pred, _ = preds[(model, "multi")]
    p, r, f, n = precision_recall_fscore_support(splits["multi"][3], pred, labels=range(len(CLASSES)), zero_division=0)
    rows += [{"model": model, "class": c, "precision": p[k], "recall": r[k], "f1": f[k], "support": n[k]}
             for k, c in enumerate(CLASSES)]
    rows.append({"model": model, "class": "macro avg", "precision": p.mean(), "recall": r.mean(), "f1": f.mean(), "support": n.sum()})

per_class_test = pd.DataFrame(rows).set_index(["model", "class"])
per_class_test.round(3)

Precision, recall and F1 of each class on the test set, with zero for undefined values (the binary table above reports class 1 only).

- **RF:** PWF and OSF are always found (recall 1.000) and HDF almost always (26 of 27, recall 0.963). OSF has the lowest precision of the three (0.800), because 6 rows are wrongly predicted as OSF: 4 with no failure, 1 TWF and 1 HDF.
- **MLP:** weaker than the RF on HDF, PWF and OSF (F1 0.820, 0.800, 0.870). HDF loses precision (0.735: 8 normal rows predicted as HDF), while PWF and OSF lose recall (5 PWF and 3 OSF predicted as "No failure").
- **TWF:** both models find only 1 of the 11 TWF rows (F1 0.118 for both).
- **Macro F1** (0.720 MLP, 0.781 RF) is pulled down mainly by TWF. Without TWF, the other four classes average 0.95 F1 for the RF and 0.87 for the MLP.

With 11 to 27 test rows per failure type, one more or one fewer correct row changes recall by 4 to 9 points, so these per-class numbers are only indicative.

## Missed Failures Vs False Alarms

In [ ]:
change = []
for (model, task), cfg in FINAL.items():
    _, X_te, _, y_te = splits[task]
    X_te = inputs(model, X_te, cfg)
    for label, d in [("default", 0.5), ("chosen", cfg["decision"])]:
        pred, _ = predict(fitted[(model, task)], X_te, task, d)
        fail, p_fail = (y_te > 0).to_numpy(), pred > 0
        change.append({"model": model, "task": task, "decision": f"{label} ({d})",
                       "missed_failures": int((fail & ~p_fail).sum()), "false_alarms": int((~fail & p_fail).sum())})
pd.DataFrame(change)

Here the same trained models are scored at the default 0.5 and at the decision value chosen on validation. This shows what the threshold choice did on test.\
Lowering the threshold trades false positives for fewer false negatives:
- **MLP binary:** 51 → 34 missed failures and 1 → 4 false alarms. 
- **RF multi:** 13 → 9 missed and 3 → 13 false alarms. 
- **MLP multi:** 24 → 20 missed and 8 → 18 false alarms. 

For RF binary, 0.4 and 0.5 give the same counts, because its probabilities are well separated around those values.
> Since we assumed a missed failure costs more than an inspection, these trades are acceptable.

## Error Analysis

### Binary Classification

In [ ]:
raw = load_raw()
_, X_te, _, y_te = splits["binary"]
(pred_mlp, p_mlp), (pred_rf, p_rf) = preds[("MLP", "binary")], preds[("RF", "binary")]
err = raw.loc[X_te.index].assign(y=y_te, p_mlp=p_mlp.round(3), p_rf=p_rf.round(3), mlp=pred_mlp, rf=pred_rf)

display(pd.DataFrame({"MLP missed": err[(err.y == 1) & (err.mlp == 0)][MODES].sum(),
                      "RF missed":  err[(err.y == 1) & (err.rf == 0)][MODES].sum(),
                      "all test failures": err[err.y == 1][MODES].sum()}))

show = ["UDI", "Type", *RAW_FEATURES, *MODES, "p_mlp", "p_rf", "mlp", "rf"]
display(err[(err.y == 1) & ((err.mlp == 0) | (err.rf == 0))][show])   # missed failures
display(err[(err.y == 0) & ((err.mlp == 1) | (err.rf == 1))][show])   # false alarms

> **Note:** The table counts modes, so a row with two modes is counted twice.

**Most of the missed failures are TWF:** 12 of 13 TWF failures are missed by both models, because tool-wear failures happen at a random point in a wear range that many healthy machines also reach. 

The 2 failures with no recorded mode are also missed by both, as expected, since nothing in the features explains them.\
Beyond that: 
- The RF misses only 1 HDF 
- The MLP misses 12 HDF, 4 PWF and 5 OSF, which are failures defined by sharp rules on temperature difference, power and strain. 

Trees capture threshold rules like these directly, while the MLP has to approximate them with smooth functions. The RF's 4 false positives all fall in conditions similar to real HDF failures (high air temperature, low speed).

### Multi-class Classification

In [ ]:
_, X_te, _, y_te = splits["multi"]
raw_te = raw.loc[X_te.index]
print("multi-mode rows in test:", int((raw_te[PRIORITY].sum(axis=1) > 1).sum()))
for model in ["MLP", "RF"]:
    pred, _ = preds[(model, "multi")]
    wrong = pred != y_te.to_numpy()
    other = np.array([p > 0 and raw_te.iloc[i][CLASSES[p]] == 1 for i, p in enumerate(pred)]) & wrong
    print(f"{model}: {other.sum()} of {wrong.sum()} test errors predicted another mode that is also present")

The test set has 5 rows with more than one active mode, which were labelled with their rarest mode. Only 1 of the MLP's 39 test errors, and none of the RF's 24, predict another mode that is also active on that row. So the labelling rule for multi-mode rows has almost no effect on the results, and nearly all errors are real misses or false positives.

## AI use

**Tools:**
- **Claude / Claude Code** (Anthropic)
- **GitHub Copilot** (GitHub)
- **Google AI search** (Google)

**How we checked the generated material**

- **Code:** every suggestion was read before it was used.
- **Text:** every number in the markdown and in the report was checked against the notebook outputs and the files in `results/`.
- **Literature:** each reference was opened by us, and the numbers in the comparison table were taken from the papers themselves
- All modelling choices are ours and are justified in the notebooks. AI output was used as a suggestion and never accepted without checking.

## Contributions

Each member owned one stream end to end (see `documentation/project_plan.md`). All three took part in the model-selection meeting, wrote the report and can explain the whole project.

| Member | Stream | Main contributions |
| --- | --- | --- |
| **José Antunes** (2023211288) | Data and final evaluation | Project setup and environment; data quality and label checks; preprocessing and engineered features (`src/data.py`); stratified splits and saved indices (`src/splits.py`); frozen final configurations (`results/best_params.json`); final test evaluation and error analysis (notebook 05); README. |
| **Nuno Leal** (2026145552) | Random Forest and context | EDA and figures (notebook 01); feature selection (notebook 02); Random Forest for the binary and multiclass tasks (notebook 03, `src/models_rf.py`); literature comparison. |
| **Adriel Lima** (2026144080) | Neural network | Shared evaluation code with repeated CV, threshold and cut-off tables and the tuning log (`src/evaluate.py`); MLP for the binary and multiclass tasks (notebook 04, `src/models_mlp.py`). |
| **All three** | | Model-selection meeting, results report and defence preparation. |